# VisWord 01 — Prefetch data to Drive

Downloads the `Tevatron/wiki-ss-corpus` to `MyDrive/VISWORD/data/wiki_ss/` plus anchors to `wiki_ss_anchors/`. Produces the exact same manifest schema as the VALAR `prefetch.py`, so all downstream notebooks/scripts are drop-in compatible.

**Runtime:** CPU or any GPU (GPU unused).  **Wallclock:** ~30–90 min for 100k rows, ~4–6 h for 500k.

Colab has unrestricted internet to HF so prefetch works normally here.

In [ ]:
# Re-establish session state (in case this is a fresh Colab runtime).
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['HF_DATASETS_CACHE'] = f'{PROJECT}/hf_cache/datasets'
os.environ['HF_HUB_CACHE'] = f'{PROJECT}/hf_cache/hub'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
%cd $REPO_DIR

## Pick target size

Our VALAR cache has 15k. Target here depends on how much Colab wallclock you can afford:
- 100k rows ≈ 1–2 h on Colab
- 500k rows ≈ 4–8 h (one long session; Drive persists so you can resume)

In [ ]:
TARGET_ROWS = 100_000  # change to 500_000 for full scale-up run

## 1 — wiki-ss-corpus (images + titles + texts)

Uses `visword.data.prefetch.prefetch_wiki_ss` (same code as VALAR). Resume-safe: if you re-run after a timeout, it continues from the last checkpointed row.

In [ ]:
from pathlib import Path
from visword.data.prefetch import prefetch_wiki_ss

cache = Path(os.environ['DATA_DIR']) / 'wiki_ss'
summary = prefetch_wiki_ss(cache, TARGET_ROWS, resume=True)
print('wiki-ss prefetch:', summary)

## 2 — Anchor triplets (small, ~30 MB)

In [ ]:
from visword.data.prefetch import prefetch_anchors
anchors_cache = Path(os.environ['DATA_DIR']) / 'wiki_ss_anchors'
print('anchors prefetch:', prefetch_anchors(anchors_cache))

## 3 — Integrity check

In [ ]:
import json
m = json.load(open(cache / 'manifest.json'))
print('wiki_ss rows:', m['num_rows'])
print('first row keys:', list(m['rows'][0].keys()))
print('sample title:', m['rows'][0]['title'])
print('sample text_path:', m['rows'][0]['text_path'])

from visword.data.manifest import verify_fingerprint
print('fingerprint OK:', verify_fingerprint(cache))

## 4 — Optional: parallel speed-up via split slicing

If `prefetch_wiki_ss` (serial streaming) is too slow, run multiple split-slice workers in parallel processes. Colab's single-VM Python can't truly parallelise CPU download, so this is most useful with `multiprocessing` — skip unless you see <1000 rows/min.

In [ ]:
# Uncomment to try the split-slice path in this notebook (requires restarting the runtime after streaming prefetch).
# from visword.data.prefetch import prefetch_split_slice
# summary = prefetch_split_slice(cache, split_start=100000, split_end=200000, idx_base=1_000_000, target_rows=100_000)
# print(summary)

## Next step

→ `02_zeroshot_vision.ipynb` (zero-shot DINOv2 / CLIP / DINO-v1 / ImageNet-ViT)
→ `03_zeroshot_text_multimodal.ipynb` (BERT / MiniLM / CLIP-text / cross-modal)

Both evaluate on the cache you just downloaded.